### Part 2 of tool calling

##### In part1 of the tool calling code send LLM the request and request for tool calling, which code identifies and does actual tool calling to send notificdation via Pushover.
##### Actually code called the notification based on information sent by LLM, but LLM didn't know that notification was sent to the sender.
##### part2 of th tool calling will inform back to LLM about the action of the code that the tool calling has been completed. and also will review the response of LLML .

In [47]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display
import json

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception ("API key is missing.")
else:
    print(OPENAI_API_KEY[:8])



sk-proj-


### Setup PushOver (send phone notification)

In [48]:
# Step 1 - setup an account in PushOver
# Step 2- Setup the app on our iphone / android phone, log into the same account
# step 3 - create an "Application/Api Token" from the browser
# step 4 - copy your user key and api token into the .env file and save the changes
# e.g PUSHOVER_USER= xxxxxxxxx PUSHOVER_TOKEN=yyyyyyyyy
# PUSHOVER_USER=uj2zzdtmw5jhsyhrenptcbut58wgzs
# PUSHOVER_TOKEN=a17pqtdb46zrxux3tughsdkykbqcgd

load_dotenv()
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

print(pushover_user)
print(pushover_token)

uj2zzdtmw5jhsyhrenptcbut58wgzs
a17pqtdb46zrxux3tughsdkykbqcgd


In [49]:
import requests

def send_notification(message:str):
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [50]:
send_notification ("Hello, I'm in a class of AI Engineering training")

### Use pushover using LLM tool

In [51]:
send_notification_function = {

    "name" : "send_notification",
    "description": "Sends a push notification ot the user's phone via pushover. Use this to alert the user about the change",
    "parameters": {
        "type": "object",
        "properties": {

            "message": {
                "type": "string",
                "description": "The notification message to send to the user's device"
            }
        },
        "required":["message"]
    }
}

### Add Pushover to the list of tools for the LLM (so far only one tool)

In [60]:
tools = [{"type": "function", "function": send_notification_function}]

### New Addition. Earlier we had one tool function (send_notification). Now adding another one
### Create a new tool function, describe it and add it to the list of tools.
### 

In [61]:
import random

#Simulates rolling  a single six-sided one
def dice_roll():
    result = random.randint(1,6)
    return result

#DESCRIBE FUNCTION FORM LLM
dice_roll_function = {

    "name" : "dice_roll",
    "description": "simulate rolling a dice to get a random number between 1 to 6. Use this when user wants to roll a dice and get a result",
    "parameters": {
        "type": "object",
        "properties": {},
        "required":[]
    }
}

# add function to teh list of tools of LLM
tools.append({"type": "function", "function": dice_roll_function})
print(tools)

[{'type': 'function', 'function': {'name': 'send_notification', 'description': "Sends a push notification ot the user's phone via pushover. Use this to alert the user about the change", 'parameters': {'type': 'object', 'properties': {'message': {'type': 'string', 'description': "The notification message to send to the user's device"}}, 'required': ['message']}}}, {'type': 'function', 'function': {'name': 'dice_roll', 'description': 'simulate rolling a dice to get a random number between 1 to 6. Use this when user wants to roll a dice and get a result', 'parameters': {'type': 'object', 'properties': {}, 'required': []}}}]


In [ ]:
# handle tool call
# added later - mutliple tool calls to same tool and later to a different tools
def handle_tool_call(tool_calls):
    # .....
    # return what to add to our "coontext" about the tool call results, a dictionary

    tool_results = []

    for tool_call in tool_calls:
        function_name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)
        #send notification
        if(function_name == "send_notification"):
            result = send_notification(args["message"])
            content = f"Notification sent successfully: {result}"
            # print(content)
        elif function_name == "dice_roll":
            content = f"Rolled: {dice_roll()}"
            # print(content)
        else:
            content = f"Unknown function: {function_name}"
            # print(content)

        print(content)
        tool_call_result = {
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": tool_call.function.name,
                "content": content
            }
        tool_results.append(tool_call_result)

    return tool_results

### CALLING THE TOOL FROM AN LLM

In [ ]:
client = OpenAI()
messages = [
        { "role":"user", "content": "I would like to roll the dice twice and send a push notificaiton about the highest roll dice number"}
    ]

response = client.chat.completions.create(
    messages = messages,
    model = "gpt-4.1-mini",
    tools = tools
)

#check if model wants to call a tool
message = response.choices[0].message
# print(message)
while message.tool_calls:
    #....handle tool call
    #....add message to "context", i.e. messages
    # .... add info about tool call response to "context", i.e. messages
    # ...invoke the LLM again to get its updated with the new response
    # ... print(message.content) from the updated LLM response

    from pprint import pprint
    pprint(message.tool_calls)

    tool_results = handle_tool_call(message.tool_calls) # send list of tool calls
   
    messages.append(message)
    # messages.append(tool_call_result)
    # change from append to extend to add a new list to an existing list
    messages.extend(tool_results) 
    print(f"before second LLM call. Tool results: {tool_results} ")

    response2 = client.chat.completions.create(
        messages = messages,
        model = "gpt-4.1-mini",
        tools = tools
    )

    #check if model wants to call a tool
    message = response2.choices[0].message
    print(f"while: {message.content}")
    
    # note: to add a safeguard or protection to avoid while going to an infinte loop for a production code.

    
print(f"***: {message.content}")


[ChatCompletionMessageFunctionToolCall(id='call_rD1oRMCWVQkFLBHabkaxLLKu', function=Function(arguments='{}', name='dice_roll'), type='function'),
 ChatCompletionMessageFunctionToolCall(id='call_z6WClef83H1SOt6cLVcLIa28', function=Function(arguments='{}', name='dice_roll'), type='function')]
Rolled: 4
Rolled: 5
before second LLM call. Tool results: [{'role': 'tool', 'tool_call_id': 'call_rD1oRMCWVQkFLBHabkaxLLKu', 'name': 'dice_roll', 'content': 'Rolled: 4'}, {'role': 'tool', 'tool_call_id': 'call_z6WClef83H1SOt6cLVcLIa28', 'name': 'dice_roll', 'content': 'Rolled: 5'}] 
while: None
[ChatCompletionMessageFunctionToolCall(id='call_xE413Ss8LTBmp0jhgCXG7uht', function=Function(arguments='{"message":"The highest rolled dice number is 5."}', name='send_notification'), type='function')]
Sent notification: The highest rolled dice number is 5.
before second LLM call. Tool results: [{'role': 'tool', 'tool_call_id': 'call_xE413Ss8LTBmp0jhgCXG7uht', 'name': 'send_notification', 'content': 'Notifica